In [1]:
from FotR import FRODO
db = FRODO(
    root_dir = '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_extended/outputs/',
    format = 'PYLOM',
    file = 'CADGroup_3_completo_stage_0.h5')

db.extract_inputs(
    keys_inputs={
        'ptos': 'xyz',    # coordenadas del mallado → data_dict['inputs']['ptos']
        'aoa': 'aoa',   # variable paramétrica   → data_dict['inputs']['aoa']
        'mach': 'M',   # variable paramétrica   → data_dict['inputs']['M']
    },
    keys_aux={},
)

db.extract_outputs(
    keys_outputs={'cp': 'BoundaryValues_CoefPressure'}  # field del Dataset → data_dict['outputs']['cp']
)

0 Warning! Import - NVTX not present!
/home/m.jaraiz/miniconda/envs/envkan_nvidia/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




[PYLOMReader] Dataset loaded in 1.400 s
Dataset of 9 variables:
  > CD  - max = 0.099306684854712, min = 0.0004836486912827
  > CL  - max = 0.5718504768239702, min = 0.0020063200116849
  > CMy  - max = 0.0094466424484941, min = -0.0762847356073729
  > M  - max = 1.489529013633728, min = 0.3043437898159027
  > Re  - max = 2127846.6018327894, min = 315933.6875
  > aoa  - max = 4.983091831207275, min = 0.0221543479710817
  > varCD  - max = 0.0245495196644242, min = 9.185734449770632e-09
  > varCL  - max = 0.0875641846286219, min = 2.6563933456150924e-06
  > varCM  - max = 0.0035326103784482, min = 9.989066188103632e-08
and 29 fields with 13862 points:
  > AugStateGrad_DensityGradient_interp - max = 0.0, min = 0.0, avg = 0.0
  > AugStateGrad_EnergyStagnationDensityGradient_interp - max = 0.0, min = 0.0, avg = 0.0
  > AugStateGrad_MomentumXGradient_interp - max = 0.0, min = 0.0, avg = 0.0
  > AugStateGrad_MomentumYGradient_interp - max = 0.0, min = 0.0, avg = 0.0
  > AugStateGrad_Momentum

In [2]:
# Acceder a los datos
xyz = db.sets.get_xyz()           # (npoints, 3)
aoa   = db.sets.get_variable('aoa') # (500,)
mach = db.sets.get_variable('mach')   # (500,)
cp  = db.sets.get_field('cp')     # (npoints, 500)

from FotR import SAM
import torch
xyz_sort, order_sort = SAM.Weapons.sort_by_centroid(xyz)
cp_sort = cp[order_sort, :]


for sep in [20,]:#range(220, 20, -2):
    tensor_ptos = torch.from_numpy(xyz_sort[::sep, :])
    tensor_cp   = torch.from_numpy(cp_sort[::sep,  :])
    N = tensor_ptos.shape[0]

    # ── derivada por longitud de arco ─────────────────────────────────────────
    dcp_ds = torch.zeros(tensor_cp.shape, dtype=torch.float64)
    for case in range(tensor_cp.shape[1]):
        dcp_ds[:, case] = SAM.Weapons.surface_derivative(
            X=tensor_ptos, f=tensor_cp[:, case], order=1
        )
        
    db_one = db.copy()
    db_one.sets.add_aux(
        array_name = 'dcp_ds',
        array = dcp_ds.numpy(),
        notes = 'Derivada sobre contorno')
    
    display(db_one.df_data)
    

AttributeError: 'FRODO' object has no attribute 'df_data'

In [3]:
db_one.sets.create_jset(verbose=True)

points_repeated: torch.Size([1386200, 3])
flcc_expanded:   torch.Size([1386200, 2])
aux original dim 2 -> flattened torch.Size([69400, 1])
out original torch.Size([13862, 100, 1]) -> flattened torch.Size([1386200, 1])


RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 1386200 but got size 69400 for tensor number 2 in the list.

#### Calculation of derivates in isolated case

In [ ]:
import matplotlib.pyplot as plt
import io
from PIL import Image


scale = 0.7
case_gif = 50

# ── Precomputar límites globales para ejes fijos ──────────────────────────────
# Se usan el subconjunto más denso (sep mínimo) como referencia
_ref_ptos = torch.from_numpy(xyz_sort[::20, :])
_ref_cp   = torch.from_numpy(cp_sort[::20, :])
_ref_dcp  = SAM.Weapons.surface_derivative(X=_ref_ptos, f=_ref_cp[:, case_gif], order=1)

Y_CP_MIN  = float(_ref_cp[:, case_gif].min())
Y_CP_MAX  = float(_ref_cp[:, case_gif].max())
Y_DCP_MIN = float(_ref_dcp.min())
Y_DCP_MAX = float(_ref_dcp.max())
X_MIN     = float(_ref_ptos[:, 0].min())
X_MAX     = float(_ref_ptos[:, 0].max())
Y_GEO_MIN = float(_ref_ptos.min()) * scale
Y_GEO_MAX = float(_ref_ptos.max()) * scale
# ─────────────────────────────────────────────────────────────────────────────

frames: list[Image.Image] = []

for sep in range(220, 20, -2):
    # tensor_ptos = torch.from_numpy(xyz_sort[::sep, :])
    # tensor_cp   = torch.from_numpy(cp_sort[::sep,  :])
    # N = tensor_ptos.shape[0]

    # # ── derivada por longitud de arco ─────────────────────────────────────────
    # dcp_ds = torch.zeros(tensor_cp.shape, dtype=torch.float64)
    # for case in range(tensor_cp.shape[1]):
    #     dcp_ds[:, case] = SAM.Weapons.surface_derivative(
    #         X=tensor_ptos, f=tensor_cp[:, case], order=1
    #     )
    # ─────────────────────────────────────────────────────────────────────────

    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    fig.patch.set_facecolor('white')

    # --- geometría (puntos de la curva) --------------------------------------
    # ax.scatter(tensor_ptos[:, 0], tensor_ptos[:, 2], c='black', s=3, zorder=3)
    # ax.set_ylabel('y', fontsize=11)
    # ax.set_xlabel('x', fontsize=11)
    # ax.set_xlim(X_MIN, X_MAX)
    # ax.set_ylim(Y_GEO_MIN, Y_GEO_MAX)

    # --- cp (eje principal) ---------------------------------
    # ax.scatter(
    #     x=tensor_ptos[:, 0],
    #     y=tensor_cp[:, case_gif],
    #     c='black',
    #     s=10,
    #     alpha=0.5,
    #     label='cp',
    # )
    ax.plot(
        tensor_ptos[:, 0],
        tensor_cp[:, case_gif],
        c = 'black',
        linewidth = 1,
    )
    ax.set_ylabel('y', fontsize=11)
    ax.set_xlabel('x', fontsize=11)
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_ylim(Y_CP_MIN - 0.05 * abs(Y_CP_MIN), Y_CP_MAX + 0.05 * abs(Y_CP_MAX))
    ax.set_yticks([])

    # --- cp (eje derecho secundario, oculto) ---------------------------------
    # ax1 = ax.twinx()
    # ax1.scatter(
    #     x=tensor_ptos[:, 0],
    #     y=tensor_cp[:, case_gif],
    #     c='black',
    #     s=10,
    #     alpha=0.5,
    #     label='cp',
    # )
    # ax1.set_ylim(Y_CP_MIN - 0.05 * abs(Y_CP_MIN),
    #              Y_CP_MAX + 0.05 * abs(Y_CP_MAX))
    # ax1.set_yticks([])

    # --- dcp/ds (eje derecho principal) --------------------------------------
    ax2 = ax.twinx()
    ax2.spines['right'].set_position(('outward', 0))   # superpuesto a ax1

    y  = dcp_ds[:, case_gif]
    n  = len(y)
    # mitad izquierda → rojo  |  mitad derecha → azul  (cara presión / succión)
    colors = ['red'] * (n // 2) + ['blue'] * (n - n // 2)

    ax2.scatter(
        x=tensor_ptos[:, 0],
        y=y,
        c=colors,
        s=15,
        alpha=0.8,
        zorder=4,
    )
    ax2.axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
    ax2.set_ylabel('dcp/ds', color='crimson', fontsize=11)
    ax2.tick_params(axis='y', colors='crimson')
    ax2.set_ylim(Y_DCP_MIN * 1.1, Y_DCP_MAX * 1.1)
    # ax2.set_yscale('symlog', linthresh=1e-2)

    # --- título con info del frame -------------------------------------------
    ax.set_title(
        f'case {case_gif}  ·  sep = {sep}  ·  N = {N} pts',
        fontsize=12, pad=8
    )

    plt.tight_layout()

    # ── capturar frame en memoria → PIL Image ─────────────────────────────────
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    frames.append(Image.open(buf).copy())   # .copy() libera el buffer
    plt.close(fig)
    buf.close()
    # ─────────────────────────────────────────────────────────────────────────

# ── Guardar GIF ───────────────────────────────────────────────────────────────
gif_name = f'dcp_ds_case{case_gif}_evolution.gif'
gif_path = './pictures/'

import os
os.makedirs(gif_path, exist_ok=True)

frames[0].save(
    os.path.join(gif_path, gif_name),
    save_all=True,
    append_images=frames[1:],
    duration=200,          # ms por frame  → ajusta a tu gusto
    loop=0,                # 0 = bucle infinito
    optimize=False,
)

print(f'GIF guardado en: {gif_path}  ({len(frames)} frames)')
# 

#### Calculation of clusters depending on the filter in airfoil